# core

> Fill in a module description here

In [1]:
# | default_exp gradio_app

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [3]:
import gradio as gr
from product_horse.api import (
    create_and_save_schema,
    extract_data_given_users,
    save_files_and_transcriptions,
    get_user_names_and_transcript_counts,
    make_video,
)
from product_horse.db import SqlModelDatabase
from product_horse.filesystems import LocalFileSystem
from product_horse.core import InMemoryQueue
from product_horse.extraction import AIModelClient, ModelType
import os
from uuid import UUID
import dotenv

In [4]:
dotenv.load_dotenv()

# putting the app working directory at ~/.product_horse/
app_working_directory = os.path.expanduser("~/.product_horse/")
if not os.path.exists(app_working_directory):
    os.makedirs(app_working_directory)
    os.makedirs(app_working_directory + "files/")


# below url is for HOVER-only.
# db = SqlModelDatabase(database_url="postgresql://localhost:5432/product_horse")
db = SqlModelDatabase(database_url=os.getenv("DATABASE_URL"))
fs = LocalFileSystem(base_path=app_working_directory + "files/")
ai_model_client = AIModelClient(model_type=ModelType.CLAUDE_3_HAIKU)
queue = InMemoryQueue()


async def save_schema(schema_text: str):
    schema = await create_and_save_schema(schema_text, db, ai_model_client)
    return schema

async def extract_info(
    user_ids: list[str],
):
    output_file_path = await extract_data_given_users(user_ids, db, ai_model_client, app_working_directory)
    return gr.File(value=output_file_path, label="Download CSV")


def refresh_user_list() -> gr.CheckboxGroup:
    # Update the choices of the CheckboxGroup -- seems like there's something weird with the get_user_names... type but ignore for now.
    return gr.CheckboxGroup(
        label="User IDs",
        choices=get_user_names_and_transcript_counts(db), # type: ignore
        interactive=True,
    )

def update_user_selection(select_all: bool) -> list[str]:
    if select_all:
        return [option[1] for option in get_user_names_and_transcript_counts(db)]
    else:
        return []


async def fetch_utterances(query: str, user_ids: list[str]) -> gr.File:
    video_path = await make_video(query, user_ids, db)
    return gr.File(value=video_path, label="Download Video")


with gr.Blocks() as app:
    with gr.Tabs() as tabs:
        with gr.TabItem("File Upload and Transcription"):
            user_id_input = gr.Textbox(label="User ID")
            user_name_input = gr.Textbox(label="User Name")
            file_input = gr.File(label="Files", file_count="multiple")
            output_text = gr.Textbox(label="Output")
            save_button = gr.Button("Save Files and Transcriptions")
            save_button.click(
                save_files_and_transcriptions,
                inputs=[user_id_input, user_name_input, file_input],
                outputs=output_text,
            )
        with gr.TabItem("Edit User Files"):
            gr.Markdown("## Edit User Files")
            
            user_dropdown = gr.Dropdown(label="Select User", choices=get_user_names_and_transcript_counts(db))

            def fetch_files_for_user(user_id: str):
                transcriptions = db.get_transcriptions(user_id)
                files_info = [(f"{transcription.file_metadata.file_name} - {transcription.file_metadata.file_path}", transcription.file_metadata.id) for transcription in transcriptions]
                return gr.Dataframe(value=files_info, headers=["File Name and Path", "File ID"], type="array")
            
            files_display = gr.Dataframe()
            user_dropdown.change(fetch_files_for_user, inputs=user_dropdown, outputs=files_display)
            
            new_user_name_input = gr.Textbox(label="New User Name")
            update_button = gr.Button("Update User Name")
            
            def update_user_name(user_id: str, new_name: str):
                if user_id:
                    db.update_user(UUID(user_id), {"name": new_name})
                    new_choices = get_user_names_and_transcript_counts()
                    return gr.Markdown(value="User name updated successfully.", visible=True), gr.Dropdown(choices=new_choices)
                else:
                    new_choices = get_user_names_and_transcript_counts()
                    return gr.Markdown(value="User not found.", visible=True), gr.Dropdown(choices=new_choices)
                
            update_markdown = gr.Markdown(value="User name updated successfully.", visible=False)
            delete_button = gr.Button("Delete User", variant='stop')
            confirm_delete_button = gr.Button("Confirm Delete", visible=False)
            delete_confirmation_text = gr.Markdown(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=False)

            def prompt_delete_confirmation(user_id: str):
                if user_id:
                    return gr.Button("Confirm Delete", visible=True), gr.Textbox(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=True)
                else:
                    return gr.Button("Confirm Delete", visible=False), gr.Textbox(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=False)

            def delete_user(user_id: str):
                db.delete_user(UUID(user_id))
                new_choices = get_user_names_and_transcript_counts()
                return gr.Dropdown(choices=new_choices), gr.Button("Confirm Delete", visible=False), gr.Markdown(value="User deleted successfully.", visible=True)

            delete_button.click(prompt_delete_confirmation, inputs=[user_dropdown], outputs=[confirm_delete_button, delete_confirmation_text])
            confirm_delete_button.click(delete_user, inputs=[user_dropdown], outputs=[user_dropdown, confirm_delete_button, delete_confirmation_text])
            update_button.click(update_user_name, inputs=[user_dropdown, new_user_name_input], outputs=[update_markdown, user_dropdown])
                        
        with gr.TabItem("Schema Definition"):
            gr.Markdown("## Schema Definition")
            schema_questions_input = gr.Textbox(label="Questions")
            schema_save_button = gr.Button("Create Schema")
            schema_save_output = gr.Textbox(label="Output")
            schema_save_button.click(
                save_schema,
                inputs=[schema_questions_input],
                outputs=schema_save_output,
            )
        with gr.TabItem("Information Extraction"):
            gr.Markdown("## Information Extraction")
            select_all_checkbox = gr.Checkbox(label="Select All", value=False)
            user_ids_input = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(db),
                interactive=True,
            )
            refresh_button = gr.Button("Refresh")
            refresh_button.click(refresh_user_list, outputs=user_ids_input)
            select_all_checkbox.change(
                fn=update_user_selection,
                inputs=[select_all_checkbox],
                outputs=[user_ids_input],
            )
            extract_button = gr.Button("Extract Information")
            csv_download = gr.File(label="Download CSV", interactive=False)
            extract_button.click(
                extract_info, inputs=[user_ids_input], outputs=[csv_download]
            )
        with gr.TabItem("Transcripts by User"):
            gr.Markdown("## Transcripts by User")
            user_transcript_dropdown = gr.Dropdown(label="Select User", choices=get_user_names_and_transcript_counts(db))

            def fetch_transcripts_for_user(user_id: str):
                transcriptions = db.get_transcriptions(user_id)
                transcript_choices = [(f"{transcription.id} - {transcription.created_at.strftime('%Y-%m-%d %H:%M:%S')}", transcription.id) for transcription in transcriptions]
                return gr.Dropdown(label="Select Transcript", choices=transcript_choices)

            transcript_dropdown = gr.Dropdown(label="Select Transcript")
            user_transcript_dropdown.change(fetch_transcripts_for_user, inputs=user_transcript_dropdown, outputs=transcript_dropdown)

            def display_transcript_content(transcript_id: str):
                print('transcript_id', transcript_id)
                transcript = db.get_transcription(transcript_id)
                return gr.Markdown(value=transcript.text, label="Transcript Content")

            transcript_content_display = gr.Markdown()
            transcript_dropdown.change(display_transcript_content, inputs=transcript_dropdown, outputs=transcript_content_display)

        with gr.TabItem("Query and Generate Video"):
            gr.Markdown("## Query and Generate Video")
            gr.Markdown(
                "Enter a query to find relevant utterances and generate a video from them. Use the transcript selector above."
            )
            query_input = gr.Textbox(label="Enter Query")
            select_all_checkbox = gr.Checkbox(label="Select All", value=False)
            user_ids_input_2 = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(db),
                interactive=True,
            )
            select_all_checkbox.change(
                fn=update_user_selection,
                inputs=[select_all_checkbox],
                outputs=[user_ids_input_2],
            )
            fetch_button = gr.Button("Create Video")
            video_download = gr.File(label="Download Video", interactive=False)
            fetch_button.click(
                fetch_utterances,
                inputs=[query_input, user_ids_input_2],
                outputs=video_download,
            )

app.queue().launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [5]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore

IMPORTANT: You are using gradio version 4.23.0, however version 4.29.0 is available, please upgrade.
--------
